In [1]:
# 환경변수 불러오기
import os
from dotenv import load_dotenv

load_dotenv()



True

In [ ]:
github_pat = os.getenv("GITHUB_PAT")

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient




if not github_pat:
  raise ValueError("GITHUB_PAT 환경변수가 없습니다.")

# 공식 문서 -> 에러 난다.
# Autorization 토큰을 Bearer에 넣어서 보내야 함
mcp_client = MultiServerMCPClient({
    "github": {
      "url": "https://api.githubcopilot.com/mcp/",
      "headers": {
        "Authorization": f"Bearer {github_pat}"
      },
      "transport" : "streamable_http"
    }
})


# client에 넣어서 보낼 데이터 중간에 "#" 이런 주석 처리 있으면 안됨 -> 주의
# mcp_client = MultiServerMCPClient({
#     "github": {
#         "url": "https://api.githubcopilot.com/mcp/",
#         "transport": "streamable_http",
#         "headers": {
#             "Authorization": f"Bearer {github_pat}"
#         }
#     }
# })

In [ ]:
tool_list = await mcp_client.get_tools()  # 이 도구들을 활용해서 agent에게 전달

In [12]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-4o-mini")  # 답변이 잘 안나오는데는 mini 모델을 써서 그럴수도 있다. (원래 gpt-4o 사용)

In [16]:
# from langgraph.prebuilt import create_agent
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tool_list,   
    system_prompt = ("Use the tools provided to you to answer the user's question")
)

# system_prompt 가 생각보다 간단하다는 것을 볼 수 있다.
# 그 이유는, 도구 그 자체에 prompt 가 상세하게 작성되어있다. (도구의 이름, 도구 설명 등등) - 에이전트에 장황한 프롬프트를 쓰지 않아도, 잘작동한다.



In [ ]:
# 
async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:

            key = list(chunk.keys())[0]
            
            if key == 'agent':
                # Agent 메시지의 내용을 가져옴. 메세지가 비어있는 경우 어떤 도구를 어떻게 호출할지 정보를 가져옴
                content = chunk['agent']['messages'][0].content if chunk['agent']['messages'][0].content != '' else chunk['agent']['messages'][0].additional_kwargs
                print(f"'agent': '{content}'")
            
            elif key == 'tools':
                # 도구 메시지의 내용을 가져옴
                for tool_msg in chunk['tools']['messages']:
                    print(f"'tools': '{tool_msg.content}'")
            
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error processing stream: {e}")
        return results

In [ ]:
from langchain_core.messages import HumanMessage

# 프롬프트를 작성
query = f"""
깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요.

PR URL: 

"""
stream_generator = agent.astream({'messages' : [HumanMessage(content=query)]})

all_chunks = await process_stream(stream_generator)

if all_chunks:
    final_result = all_chunks[-1]
    print(final_result)


- stdio 형식으로도 작성할 수 있다.